In [63]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [64]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

#Vector stores 
from langchain_community.vectorstores import Chroma
#Utility Imports 
import numpy as np 
from typing import List

In [9]:
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]
sample_docs

['\n    Machine Learning Fundamentals\n    \n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n    \n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    eff

In [10]:
import tempfile
temp_dir = tempfile.mkdtemp()
local_dir="/Users/vijayabhaskarv/ml-projects/rag/Rag_Krish/0-DataIngestionParsing/data/"
for i,doc in enumerate(sample_docs):
    with open(f"{local_dir}/doc_{i}.txt",'w') as f:
        f.write(doc)
        print(f"director written is : {local_dir}/doc_{i}.txt")
    

director written is : /Users/vijayabhaskarv/ml-projects/rag/Rag_Krish/0-DataIngestionParsing/data//doc_0.txt
director written is : /Users/vijayabhaskarv/ml-projects/rag/Rag_Krish/0-DataIngestionParsing/data//doc_1.txt
director written is : /Users/vijayabhaskarv/ml-projects/rag/Rag_Krish/0-DataIngestionParsing/data//doc_2.txt


### Document Loading

In [11]:
from langchain_community.document_loaders import DirectoryLoader,TextLoader
#load documents from directory 
loader = DirectoryLoader( "/Users/vijayabhaskarv/ml-projects/rag/Rag_Krish/0-DataIngestionParsing/data", 
    glob="*.txt", 
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'}
)
documents = loader.load()
print(f"Loaded {len(documents)}")
print(f"First document preview: ")
print(f"{documents[0].page_content[:200]}")

Loaded 3
First document preview: 

    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named enti


### Document splitting

In [12]:
#Intialise text splitter 
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=[" "]
)
chunks= text_splitter.split_documents(documents)
print(f"created chunks of length {len(chunks)} from documents: {len(documents)}")

created chunks of length 5 from documents: 3


In [13]:
#Chunk Example:
print(f"Chunk[0] page Content(150 chars ): {chunks[0].page_content[:150]}")
print(f"Chunk[0] page Metadata is {chunks[0].metadata}")

Chunk[0] page Content(150 chars ): Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks i
Chunk[0] page Metadata is {'source': '/Users/vijayabhaskarv/ml-projects/rag/Rag_Krish/0-DataIngestionParsing/data/doc_2.txt'}


### store chunks in Vector Store locally 

In [14]:
#Create embeddings which we need to supply to the vector store 
#Access open AI key
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [15]:
sample_text = "Understanding RAG is great skill"
embeddings = OpenAIEmbeddings()
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x2976fa690>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x294c2da30>, model='text-embedding-ada-002', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [16]:
vectors = embeddings.embed_query(sample_text)

In [17]:
vectors

[-0.0064142313785851,
 7.708201883360744e-05,
 0.0226882454007864,
 -0.026395738124847412,
 0.006041467189788818,
 0.012116516940295696,
 -0.031245032325387,
 -0.009060521610081196,
 -0.03567790240049362,
 -0.04373767226934433,
 0.009174701757729053,
 0.0009898401331156492,
 -0.0024716618936508894,
 0.006897817365825176,
 -0.001354208798147738,
 0.005329521372914314,
 0.034012217074632645,
 0.013432946056127548,
 0.0006191748543642461,
 -0.008691116236150265,
 -0.0018235223833471537,
 0.0008798579219728708,
 -0.024515125900506973,
 -0.005107877776026726,
 -0.018027013167738914,
 -0.001224916777573526,
 0.03414654731750488,
 -0.01223741378635168,
 0.002449833555147052,
 -0.009302315302193165,
 0.012620252557098866,
 -0.0023675565607845783,
 0.0003857774136122316,
 -0.02157331071794033,
 -0.007462001405656338,
 -0.029659943655133247,
 -0.01162621472030878,
 -0.003915703855454922,
 0.0009948775405064225,
 -0.0013802351895719767,
 0.041239142417907715,
 0.0019544935785233974,
 0.0019847177

### Intialise ChromaDB and Store the chunks in it

In [18]:
chunks

[Document(metadata={'source': '/Users/vijayabhaskarv/ml-projects/rag/Rag_Krish/0-DataIngestionParsing/data/doc_2.txt'}, page_content='Natural Language Processing (NLP)\n    \n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
 Document(metadata={'source': '/Users/vijayabhaskarv/ml-projects/rag/Rag_Krish/0-DataIngestionParsing/data/doc_0.txt'}, page_content='Machine Learning Fundamentals\n    \n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: 

In [19]:
persist_dir = "./chroma.db"
vector_store=Chroma.from_documents(
   documents=chunks,
   embedding= OpenAIEmbeddings(),
   persist_directory=persist_dir,
   collection_name="rag_collection" 
)
print(f"Vector store created with {vector_store._collection.count()} Vectors")
print(f"Persisted to {persist_dir}")

Vector store created with 24 Vectors
Persisted to ./chroma.db


### Test Similarity Search

In [20]:
query = "What are the types of machine learning"
similarity_docs = vector_store.similarity_search(query,k=3)
similarity_docs

[Document(metadata={'source': '/Users/vijayabhaskarv/ml-projects/rag/Rag_Krish/0-DataIngestionParsing/data/doc_0.txt'}, page_content='Machine Learning Fundamentals\n    \n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': '/Users/vijayabhaskarv/ml-projects/rag/Rag_Krish/0-DataIngestionParsing/data/doc_0.txt'}, page_content='Machine Learning Fundamentals\n    \n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of mach

In [21]:
query="What is NLP?"
similarity_docs = vector_store.max_marginal_relevance_search(query,k=3)
similarity_docs

[Document(metadata={'source': '/Users/vijayabhaskarv/ml-projects/rag/Rag_Krish/0-DataIngestionParsing/data/doc_2.txt'}, page_content='Natural Language Processing (NLP)\n    \n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
 Document(metadata={'source': '/Users/vijayabhaskarv/ml-projects/rag/Rag_Krish/0-DataIngestionParsing/data/doc_1.txt'}, page_content='while Recurrent Neural Networks (RNNs) and Transformers \n    excel at sequential data processing.'),
 Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='methods, and \nActor-Critic methods. RL has been su

In [22]:
for i,doc in enumerate(similarity_docs):
    print(f"document {i+1} is {doc.page_content[:200]}")
    print(f"document source is {doc.metadata} \n")

document 1 is Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity re
document source is {'source': '/Users/vijayabhaskarv/ml-projects/rag/Rag_Krish/0-DataIngestionParsing/data/doc_2.txt'} 

document 2 is while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
document source is {'source': '/Users/vijayabhaskarv/ml-projects/rag/Rag_Krish/0-DataIngestionParsing/data/doc_1.txt'} 

document 3 is methods, and 
Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), 
robotics, and autonomous systems.
document source is {'source': 'manual_addition', 'topic': 'reinforcement_learning'} 



### Advanced similarity search with score 

In [23]:
result_scores = vector_store.similarity_search_with_score(query,k=3)
result_scores

[(Document(metadata={'source': '/Users/vijayabhaskarv/ml-projects/rag/Rag_Krish/0-DataIngestionParsing/data/doc_2.txt'}, page_content='Natural Language Processing (NLP)\n    \n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
  0.20614449679851532),
 (Document(metadata={'source': '/Users/vijayabhaskarv/ml-projects/rag/Rag_Krish/0-DataIngestionParsing/data/doc_2.txt'}, page_content='Natural Language Processing (NLP)\n    \n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment an

#### ChromaDB use Euclidean distance for similarity search.
#### Lower the score more similar  


### RAG Chain

In [24]:
from langchain_core.runnables import RunnableParallel,RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [25]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo")
llm.invoke("Whats my name?")

AIMessage(content="I'm sorry, I cannot determine your name as I am an AI digital assistant and do not have access to personal information.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 11, 'total_tokens': 36, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DNy4OruLFjWwVwQ9kwP1R8CpDTjMN', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019d2eb0-a9f1-7c90-af77-85f9f1ecf607-0', usage_metadata={'input_tokens': 11, 'output_tokens': 25, 'total_tokens': 36, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [26]:
#Convert vector store as a retriever 
retriever = vector_store.as_retriever(
    search_kwarg={"k":3}
)
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x298371070>, search_kwargs={})

In [27]:
from langchain_core.prompts import ChatPromptTemplate
system_prompt="""You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context: {context}"""
prompt= ChatPromptTemplate.from_messages([("system",system_prompt),
                                ("human","{input}") ])
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know. \nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [28]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def get_input(x):
    return x["input"]

rag_chain = (
    {"context": get_input | retriever | format_docs, "input": get_input}
    | prompt 
    | llm
    | StrOutputParser()
)

In [29]:
response = rag_chain.invoke({"input":"What is Deep Learning?"})
response

"Deep learning is a subset of machine learning that utilizes artificial neural networks inspired by the human brain's structure. It involves interconnected layers of nodes and has significantly impacted areas such as computer vision, natural language processing, and speech recognition. Convolutional Neural Networks (CNNs) are especially useful for image processing within the realm of deep learning."

### RAG LCEL With Examples

In [30]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough,RunnableParallel

In [31]:
#Create Custom Prompt 
custom_prompt = ChatPromptTemplate.from_template(""""Use the following context to answer the question. 
If you don't know the answer based on the context, say you don't know.
Provide specific details from the context to support your answer.

Context:
{context}

Question: {question}

Answer:""")
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='"Use the following context to answer the question. \nIf you don\'t know the answer based on the context, say you don\'t know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:'), additional_kwargs={})])

In [32]:
def format_docs(docs):
    return "\n\n".join( doc.page_content for doc in docs)

In [33]:
#Build the chain using LCEL 
rag_chain_lcel= (
    {
    "context": retriever | format_docs,
    "question": RunnablePassthrough()
    }
    | custom_prompt
    | llm
    | StrOutputParser()
)
rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x298371070>, search_kwargs={})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='"Use the following context to answer the question. \nIf you don\'t know the answer based on the context, say you don\'t know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:'), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x299f96e70>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x299f68650>, root_client=<openai

In [34]:
response = rag_chain_lcel.invoke("What is Deep learning? ")

In [35]:
response

'Deep learning is a subset of machine learning based on artificial neural networks. These networks consist of layers of interconnected nodes, inspired by the human brain. It has revolutionized fields like computer vision, natural language processing, and speech recognition.'

In [36]:
retriever.invoke("What is Deep learning?")

[Document(metadata={'source': '/Users/vijayabhaskarv/ml-projects/rag/Rag_Krish/0-DataIngestionParsing/data/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n    \n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and'),
 Document(metadata={'source': '/Users/vijayabhaskarv/ml-projects/rag/Rag_Krish/0-DataIngestionParsing/data/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n    \n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep 

In [37]:
def query_rag_lcel(question):
    print(f"Question: {question}")
    print("_"*50)
    
    #Method-1: Pass string directly ( Using RunnablePassthrough)
    answer = rag_chain_lcel.invoke(question)
    print(f"Answer: {answer}")
    print("\n Source Documents")
    docs = retriever.invoke(question)
    for i,doc in enumerate(docs):
        print(f"\n -----Source {i+1}-----")
        print(doc.page_content[:200] + "....")
        
        

In [38]:
query_rag_lcel("What are the key concepts in reinforcement learning?")

Question: What are the key concepts in reinforcement learning?
__________________________________________________
Answer: The key concepts in reinforcement learning are states, actions, rewards, policies, and value functions. This is supported by the specific details from the context mentioned earlier: "Key concepts in RL include: states, actions, rewards, policies, and value functions."

 Source Documents

 -----Source 1-----
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or p....

 -----Source 2-----
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or p....

 -----Source 3-----
data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.....

 -

In [39]:
vector_store

In [40]:
# Add new documents to the existing vector store
new_document = """
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or penalties 
based on its actions and learns to maximize cumulative reward over time. Key concepts 
in RL include: states, actions, rewards, policies, and value functions. Popular RL 
algorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and 
Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), 
robotics, and autonomous systems.
"""
new_document

'\nReinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL \nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and \nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.\n'

In [41]:
#Just display existing chunks that we have already
chunks

[Document(metadata={'source': '/Users/vijayabhaskarv/ml-projects/rag/Rag_Krish/0-DataIngestionParsing/data/doc_2.txt'}, page_content='Natural Language Processing (NLP)\n    \n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
 Document(metadata={'source': '/Users/vijayabhaskarv/ml-projects/rag/Rag_Krish/0-DataIngestionParsing/data/doc_0.txt'}, page_content='Machine Learning Fundamentals\n    \n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: 

In [42]:
new_doc = Document(page_content= new_document,
                   metadata={"source": "manual_addition", "topic": "reinforcement_learning"})

new_doc

Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='\nReinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL \nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and \nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.\n')

In [43]:
#Now split the documents into chunks
new_chunks = text_splitter.split_documents([new_doc])
new_chunks

[Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='Reinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL \nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and \nActor-Critic methods. RL has been'),
 Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='methods, and \nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.')]

In [44]:
vector_store.add_documents(new_chunks)

['1dcbf141-b76f-4575-a773-32979c0ca5a4',
 '48c7a6de-157c-49a9-904c-76640535cb1d']

In [45]:
print(f"Added {len(new_chunks)} new chunks to the vector store")
print(f"Total vectors now: {vector_store._collection.count()}")

Added 2 new chunks to the vector store
Total vectors now: 26


In [46]:
new_question="What are the keys concepts in reinforcement learning"
result=query_rag_lcel(new_question)

Question: What are the keys concepts in reinforcement learning
__________________________________________________
Answer: The key concepts in reinforcement learning are states, actions, rewards, policies, and value functions. 

Specific details from the context to support the answer include: "Key concepts in RL include: states, actions, rewards, policies, and value functions."

 Source Documents

 -----Source 1-----
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or p....

 -----Source 2-----
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or p....

 -----Source 3-----
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decis

### Chat history with memory

In [51]:
from langchain.chains import create_history_aware_retriever,create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain 
from langchain_core.prompts import MessagesPlaceholder,ChatPromptTemplate
from langchain_core.messages import HumanMessage,AIMessage

In [52]:
contextualize_q_system_prompt = """Given a chat history and the latest user question 
which might reference context in the chat history, formulate a standalone question 
which can be understood without the chat history. Do NOT answer the question, 
just reformulate it if needed and otherwise return it as is."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([("system",contextualize_q_system_prompt),
                                                           MessagesPlaceholder("chat_history"),
                                                           ("human","{input}")])


In [53]:
history_aware_retriever = create_history_aware_retriever(llm,retriever,contextualize_q_prompt)

In [55]:
# Create a new document chain with history
qa_system_prompt = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context: {context}"""
qa_prompt=ChatPromptTemplate.from_messages([("system",qa_system_prompt),
                                            MessagesPlaceholder("chat_history"),
                                            ("human","{input}")])

question_answer_chain= create_stuff_documents_chain(llm,qa_prompt)
conversational_rag_chain = create_retrieval_chain(history_aware_retriever,
                                                 question_answer_chain)

conversational_rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x298371070>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Anno

In [56]:
chat_history = []
#First question
result1 = conversational_rag_chain.invoke({
    "chat_history":chat_history,
    "input":"what is machine learning?"
})
print(f"Q:What is machine learning?")
print(f"A: {result1['answer']}")

Q:What is machine learning?
A: Machine learning is a subset of artificial intelligence that allows systems to learn and improve from data without being explicitly programmed. It consists of three main types: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data, unsupervised learning finds patterns in unlabeled data, and reinforcement learning learns through interactions with an environment.


In [57]:
chat_history.extend([HumanMessage(content="What is machine learning?"),
                     AIMessage(content=result1['answer'])])

In [58]:
result2=conversational_rag_chain.invoke({"chat_history":chat_history,
                                         "input":"What are its main types?"})

In [59]:
result2['answer']

'The main types of machine learning are supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data to train models, while unsupervised learning finds patterns in unlabeled data. Reinforcement learning learns through interactions with an environment to achieve a goal.'

In [61]:
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model

In [65]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [67]:
from groq import Groq

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

models = client.models.list()

for model in models.data:
    print(model.id)

openai/gpt-oss-20b
whisper-large-v3-turbo
groq/compound
whisper-large-v3
moonshotai/kimi-k2-instruct
meta-llama/llama-prompt-guard-2-22m
moonshotai/kimi-k2-instruct-0905
meta-llama/llama-4-scout-17b-16e-instruct
openai/gpt-oss-safeguard-20b
openai/gpt-oss-120b
canopylabs/orpheus-v1-english
llama-3.1-8b-instant
groq/compound-mini
canopylabs/orpheus-arabic-saudi
llama-3.3-70b-versatile
meta-llama/llama-prompt-guard-2-86m
qwen/qwen3-32b
allam-2-7b


In [70]:
llm = init_chat_model(model="groq:llama-prompt-guard-2-22m")
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x280b835c0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x280ea1190>, model_name='llama-prompt-guard-2-22m', model_kwargs={}, groq_api_key=SecretStr('**********'))